In [22]:
import pandas as pd
import os
folder_to_read = "C:\\Users\\Elouan\\Documents\\Projet Biais LLM\\Post Treatment 2.0\\files"
all_files = os.listdir(folder_to_read)
print(all_files)
all_dfs = dict()
for file in all_files:
    df = None
    if file.endswith(".csv"):
        df = pd.read_csv(os.path.join(folder_to_read, file))
    elif file.endswith(".xlsx"):
        df = pd.read_excel(os.path.join(folder_to_read, file))
    elif file.endswith(".json"):
        df = pd.read_json(os.path.join(folder_to_read, file))
    if df is not None:    
        all_dfs[file] = df
        print(f"Loaded {file} with shape {df.shape}")
        print(f"Columns: {df.columns.tolist()}")

for file, df in all_dfs.items():
    if 'Unnamed: 0' in df.columns:
        df.drop(columns=['Unnamed: 0'], inplace=True)
        print(f"Dropped 'Unnamed: 0' from {file}")


['DeepSeek-R1-Distill-Qwen-1.5B_gender_classification_merged.csv', 'DeepSeek-R1-Distill-Qwen-7B_gender_classification_merged.csv', 'desktop.ini', 'Gemma-3-12b-it_gender_classification_merged.csv', 'Llama-3.1-8B-Instruct_gender_classification_merged.csv', 'Ministral-8B-Instruct-2410_gender_classification_merged.csv', 'Qwen-2.5-7B-Instruct_gender_classification_merged.csv']
Loaded DeepSeek-R1-Distill-Qwen-1.5B_gender_classification_merged.csv with shape (10808, 14)
Columns: ['index', 'song_title', 'artist', 'original_gender', 'original_continent', 'source', 'lyrics', 'predicted_gender', 'predicted_continent', 'gender_keywords', 'gender_reasoning', 'continent_keywords', 'continent_reasoning', 'raw_model_response']
Loaded DeepSeek-R1-Distill-Qwen-7B_gender_classification_merged.csv with shape (8106, 14)
Columns: ['index', 'song_title', 'artist', 'original_gender', 'original_continent', 'source', 'lyrics', 'predicted_gender', 'predicted_continent', 'gender_keywords', 'gender_reasoning', 'co

In [23]:
common_columns = set.intersection(*(set(df.columns) for df in all_dfs.values()))
print(f"Common columns across all DataFrames: {common_columns}")
#print per DataFrame the columns that are not in common_columns
for file, df in all_dfs.items():
    unique_columns = set(df.columns) - common_columns
    print(f"Columns in {file} not in common: {unique_columns}")

Common columns across all DataFrames: {'index', 'artist', 'continent_reasoning', 'raw_model_response', 'predicted_gender', 'continent_keywords', 'song_title', 'gender_keywords', 'gender_reasoning', 'original_continent', 'original_gender', 'source', 'lyrics', 'predicted_continent'}
Columns in DeepSeek-R1-Distill-Qwen-1.5B_gender_classification_merged.csv not in common: set()
Columns in DeepSeek-R1-Distill-Qwen-7B_gender_classification_merged.csv not in common: set()
Columns in Gemma-3-12b-it_gender_classification_merged.csv not in common: set()
Columns in Llama-3.1-8B-Instruct_gender_classification_merged.csv not in common: set()
Columns in Ministral-8B-Instruct-2410_gender_classification_merged.csv not in common: set()
Columns in Qwen-2.5-7B-Instruct_gender_classification_merged.csv not in common: set()


In [24]:
list(all_dfs.values())[0].head()
# predicted_gender	predicted_continent	gender_keywords	gender_reasoning	continent_keywords	continent_reasoning	raw_model_response
l_columns = ['predicted_gender', 'predicted_continent', 'gender_keywords', 'gender_reasoning', 'continent_keywords', 'continent_reasoning', 'raw_model_response']
common_columns = set(all_dfs['Qwen-2.5-7B-Instruct_gender_classification_merged.csv'].columns).difference(l_columns)
for name, df in all_dfs.items():
    name_model = name.split("_")[0]
    for col in l_columns:
        new_col_name = f"{name_model}_{col}"
        df.rename(columns={col: new_col_name}, inplace=True)

In [25]:
max_length = max(df.shape[0] for df in all_dfs.values())

# DeepSeek-R1-Distill-Qwen-7B_gender_classification_merged.csv has 8106 rows instead of 10808, fill the common columns with others dataframes and l_columns with NaN
#add missing rows with NaN
current_length = all_dfs['DeepSeek-R1-Distill-Qwen-7B_gender_classification_merged.csv'].shape[0]
if current_length < max_length:
    rows_to_add = max_length - current_length
    df_to_add = pd.DataFrame(index=range(rows_to_add), columns=all_dfs['DeepSeek-R1-Distill-Qwen-7B_gender_classification_merged.csv'].columns)
    all_dfs['DeepSeek-R1-Distill-Qwen-7B_gender_classification_merged.csv'] = pd.concat([all_dfs['DeepSeek-R1-Distill-Qwen-7B_gender_classification_merged.csv'], df_to_add], ignore_index=True)

for column in all_dfs['DeepSeek-R1-Distill-Qwen-7B_gender_classification_merged.csv'].columns:
    if column in common_columns:
        all_dfs['DeepSeek-R1-Distill-Qwen-7B_gender_classification_merged.csv'][column] = all_dfs['Gemma-3-12b-it_gender_classification_merged.csv'][column]

print(all_dfs['DeepSeek-R1-Distill-Qwen-7B_gender_classification_merged.csv'].shape)
all_dfs['DeepSeek-R1-Distill-Qwen-7B_gender_classification_merged.csv'].head()

(10808, 14)


,index,song_title,artist,original_gender,original_continent,source,lyrics,DeepSeek-R1-Distill-Qwen-7B_predicted_gender,DeepSeek-R1-Distill-Qwen-7B_predicted_continent,DeepSeek-R1-Distill-Qwen-7B_gender_keywords,DeepSeek-R1-Distill-Qwen-7B_gender_reasoning,DeepSeek-R1-Distill-Qwen-7B_continent_keywords,DeepSeek-R1-Distill-Qwen-7B_continent_reasoning,DeepSeek-R1-Distill-Qwen-7B_raw_model_response
0,0,Baby It's Cold Outside (feat. Christina Aguilera),CeeLo Green,Male,North America,spotify,i really cannot stay baby it is cold outside g...,female,North America,"mother, father, sister, brother, aunt, hold, d...",The lyrics contain phrases and references typi...,"cold outside, family, neighbor, tropical shore...","The lyrics reference a cold environment, famil...","Okay, so I'm trying to classify these song lyr..."
1,1,Soldier,James TW,Male,Europe,spotify,hold your breath do not look down keep trying ...,female,North America,"darling, i will not let anything stop us, figh...",The lyrics use affectionate and nurturing lang...,"O Canada, through the depths of despair, keep ...","The mention of ""O Canada"" is a strong indicato...","Alright, so I need to classify these song lyri..."
2,2,Satisfy You,Diddy,Male,North America,spotify,all i want is somebody who is going to love me...,NaN,NaN,NaN,NaN,NaN,NaN,"Okay, so I need to classify these song lyrics ..."
3,3,Tender Lover,Babyface,Male,North America,spotify,feels good everybody tender lover tender love ...,female,North America,"tender lover, tender love, Eleanor Rigby","The lyrics use phrases like ""tender lover,"" ""t...","broken heart, love, believe, sky is the limit","The themes of love, care, and belief, along wi...","Okay, so I need to classify these song lyrics ..."
4,4,Ti volevo dedicare (feat. J-AX & Boomdabash),Rocco Hunt,Male,Europe,spotify,I have something to tell you that I've been wa...,male,NaN,NaN,NaN,NaN,NaN,"Okay, so I'm trying to classify these song lyr..."


In [26]:
# Merging all dataframes on 'index' and 'song_title' columns
merged_df = None
for df in all_dfs.values():
    if merged_df is None:
        merged_df = df
    else:
        merged_df = pd.merge(merged_df, df, on=list(common_columns), how='inner')

print(merged_df.shape)

merged_df.to_csv(os.path.join(folder_to_read, "C:\\Users\\Elouan\\Documents\\Projet Biais LLM\\Post Treatment 2.0\\merged_model_responses.csv"), index=False)

(10808, 49)


In [27]:
# "C:\Users\Elouan\Documents\Projet Biais LLM\Projet-DEBIAR---Labellisation-and-hidden-states\lyrics_toxicities_wasabi.xlsx"
toxicity_1 = pd.read_excel("C:\\Users\\Elouan\\Documents\\Projet Biais LLM\\Projet-DEBIAR---Labellisation-and-hidden-states\\lyrics_toxicities_wasabi.xlsx")
# "C:\Users\Elouan\Documents\Projet Biais LLM\Projet-DEBIAR---Labellisation-and-hidden-states\lyrics_toxicities2.xlsx"
toxicity_2 = pd.read_excel("C:\\Users\\Elouan\\Documents\\Projet Biais LLM\\Projet-DEBIAR---Labellisation-and-hidden-states\\lyrics_toxicities2.xlsx")

print(toxicity_1.columns)
print(toxicity_1.shape)
print(toxicity_2.columns)
print(toxicity_2.shape)

def clean_string(s):
    if isinstance(s, str):
        for char in ['-', '_', '.', ',', ';', ':', '!', '?', '"', "'", '(', ')', '[', ']', '{', '}', '/', '\\', '|', '@', '#', '$', '%', '^', '&', '*', '~', '`']:
            s = s.replace(char, ' ')
        s = s.strip().lower()
    return s

# keys 'toxicity', 'identity_attack', 'sexually_explicit'
def find_toxicity(english_lyrics):
    d = dict()
    d['toxicity'] = pd.NA
    d['identity_attack'] = pd.NA
    d['sexually_explicit'] = pd.NA

    row1 = toxicity_1[(toxicity_1['english_lyrics'] == english_lyrics)]
    row2 = toxicity_2[(toxicity_2['english_lyrics'] == english_lyrics)]
    #remove nan rows
    row1 = row1.dropna(subset=['toxicity', 'identity_attack', 'sexually_explicit'])
    row2 = row2.dropna(subset=['toxicity', 'identity_attack', 'sexually_explicit'])
    if not row1.empty:
        d['toxicity'] = row1.iloc[0]['toxicity']
        d['identity_attack'] = row1.iloc[0]['identity_attack']
        d['sexually_explicit'] = row1.iloc[0]['sexually_explicit']
    elif not row2.empty:
        d['toxicity'] = row2.iloc[0]['toxicity']
        d['identity_attack'] = row2.iloc[0]['identity_attack']
        d['sexually_explicit'] = row2.iloc[0]['sexually_explicit']
    return d

Index(['track_name', 'track_artist', 'gender', 'country', 'continent',
       'english_lyrics', 'toxicity', 'identity_attack', 'sexually_explicit'],
      dtype='object')
(4466, 9)
Index(['track_name', 'track_artist', 'english_lyrics', 'toxicity',
       'identity_attack', 'sexually_explicit', 'genre', 'ethnicity'],
      dtype='object')
(14083, 8)


In [28]:
merged_df = pd.read_csv("C:\\Users\\Elouan\\Documents\\Projet Biais LLM\\Post Treatment 2.0\\merged_model_responses.csv")

merged_df['toxicity'] = pd.NA
merged_df['identity_attack'] = pd.NA
merged_df['sexually_explicit'] = pd.NA
for index, row in merged_df.iterrows():
    toxicity_dict = find_toxicity(row['lyrics'])
    merged_df.at[index, 'toxicity'] = toxicity_dict['toxicity']
    merged_df.at[index, 'identity_attack'] = toxicity_dict['identity_attack']
    merged_df.at[index, 'sexually_explicit'] = toxicity_dict['sexually_explicit']
    if index % 1000 == 0:
        print(f"Processed {index} rows")

print("Nb of rows with toxicity info:", merged_df['toxicity'].notna().sum())


Processed 0 rows
Processed 1000 rows
Processed 2000 rows
Processed 3000 rows
Processed 4000 rows
Processed 5000 rows
Processed 6000 rows
Processed 7000 rows
Processed 8000 rows
Processed 9000 rows
Processed 10000 rows
Nb of rows with toxicity info: 10719


In [31]:
merged_df = pd.read_csv('C:\\Users\\Elouan\\Documents\\Projet Biais LLM\\Post Treatment 2.0\\merged_model_responses.csv')
merged_df2 = pd.read_csv('C:\\Users\\Elouan\\Documents\\Projet Biais LLM\\Post Treatment 2.0\\backup\\merged_model_responses.csv')
print(f"Shape of merged_df: {merged_df.shape}, Shape of merged_df2: {merged_df2.shape}")
c1 = merged_df.columns.to_list()
c2 = merged_df2.columns.to_list()
merged_df['toxicity'] = merged_df2['toxicity']
merged_df['identity_attack'] = merged_df2['identity_attack']
merged_df['sexually_explicit'] = merged_df2['sexually_explicit']
merged_df.to_csv('C:\\Users\\Elouan\\Documents\\Projet Biais LLM\\Post Treatment 2.0\\merged_model_responses.csv', index=False)
diff = set(c1).symmetric_difference(set(c2))
print(f"Different columns between the two dataframes: {diff}")
merged_df2.head()

Shape of merged_df: (10808, 49), Shape of merged_df2: (10808, 38)
Different columns between the two dataframes: {'DeepSeek-R1-Distill-Qwen-1.5B_raw_model_response', 'DeepSeek-R1-Distill-Qwen-1.5B_predicted_gender', 'DeepSeek-R1-Distill-Qwen-1.5B_gender_keywords', 'Ministral-8B-Instruct-2410_gender_reasoning', 'Ministral-8B-Instruct-2410_predicted_gender', 'identity_attack', 'sexually_explicit', 'Ministral-8B-Instruct-2410_continent_keywords', 'Ministral-8B-Instruct-2410_continent_reasoning', 'Ministral-8B-Instruct-2410_raw_model_response', 'toxicity', 'Ministral-8B-Instruct-2410_predicted_continent', 'DeepSeek-R1-Distill-Qwen-1.5B_gender_reasoning', 'Ministral-8B-Instruct-2410_gender_keywords', 'DeepSeek-R1-Distill-Qwen-1.5B_continent_keywords', 'DeepSeek-R1-Distill-Qwen-1.5B_predicted_continent', 'DeepSeek-R1-Distill-Qwen-1.5B_continent_reasoning'}


,index,song_title,artist,original_gender,original_continent,source,lyrics,DeepSeek-R1-Distill-Qwen-7B_predicted_gender,DeepSeek-R1-Distill-Qwen-7B_predicted_continent,DeepSeek-R1-Distill-Qwen-7B_gender_keywords,...,Qwen-2.5-7B-Instruct_predicted_gender,Qwen-2.5-7B-Instruct_predicted_continent,Qwen-2.5-7B-Instruct_gender_keywords,Qwen-2.5-7B-Instruct_gender_reasoning,Qwen-2.5-7B-Instruct_continent_keywords,Qwen-2.5-7B-Instruct_continent_reasoning,Qwen-2.5-7B-Instruct_raw_model_response,toxicity,identity_attack,sexually_explicit
0,0,Baby It's Cold Outside (feat. Christina Aguilera),CeeLo Green,Male,North America,spotify,i really cannot stay baby it is cold outside g...,female,North America,"mother, father, sister, brother, aunt, hold, d...",...,female,North America,"""baby"", ""my mother"", ""my father"", ""my sister"",...","The use of terms like ""baby"" and references to...","""fireplace"", ""cabs"", ""pneumonia"", ""sister"", ""b...","The mention of a fireplace, the use of the ter...",LYRICS: i really cannot stay baby it is cold o...,0.254629,0.015556,0.193224
1,1,Soldier,James TW,Male,Europe,spotify,hold your breath do not look down keep trying ...,female,North America,"darling, i will not let anything stop us, figh...",...,female,North America,"darling, proud, get on my shoulders and carry ...",The lyrics use a lot of affectionate terms lik...,"not a soldier, get on my shoulders and carry y...",The lyrics contain references to personal stru...,LYRICS: hold your breath do not look down keep...,0.234514,0.020983,0.062000
2,2,Satisfy You,Diddy,Male,North America,spotify,all i want is somebody who is going to love me...,NaN,NaN,NaN,...,female,North America,"love, woman, your woman, your soul, my love, m...",The lyrics predominantly use terms related to ...,"money, car, diamond, baguette, first child, tr...",The lyrics reference contemporary consumer goo...,LYRICS: all i want is somebody who is going to...,0.557396,0.089973,0.411561
3,3,Tender Lover,Babyface,Male,North America,spotify,feels good everybody tender lover tender love ...,female,North America,"tender lover, tender love, Eleanor Rigby",...,female,Europe,"""tender lover"", ""eleanor rigby"", ""broken heart...","The use of terms like ""tender lover"" and ""tend...","""eleanor rigby"", ""love has no place in her hom...","The reference to ""Eleanor Rigby"" and the overa...",LYRICS: feels good everybody tender lover tend...,0.166786,0.018169,0.158406
4,4,Ti volevo dedicare (feat. J-AX & Boomdabash),Rocco Hunt,Male,Europe,spotify,I have something to tell you that I've been wa...,male,NaN,NaN,...,female,Europe,"""I have something to tell you"", ""you won't lea...",The lyrics contain several romantic and emotio...,"""Coca Cola and Malibu"", ""Let's dance like a tr...",The lyrics reference various cultural elements...,LYRICS: I have something to tell you that I've...,0.184591,0.009767,0.138095
